# 03 — Audit the frozen Wall stopper and extend evaluation

Use **your own account that ran notebook 02**, with its existing Drive results.
This notebook makes **no Astra API calls**, never changes the accepted weights,
and saves diagnostics under a separate directory. It reuses the exact runtime
embedded in notebook 02, the same project commit, and the same release assets.

Run setup, the audit, and export first. The three expensive sections are opt-in:
change their `RUN_...` switch only when ready. GPU experiments run serially; do
not run notebooks 02 and 03 simultaneously on the same runtime/experiment.

These are **post-hoc diagnostics** motivated by six observed episodes. New episodes
are held out from that pilot, on the **same fixed wall layout**; 12 episodes still
do not establish general equivalence or robustness to different layouts. Do not
send these results back to the generator to repair this controller under the
one-shot protocol.


In [ ]:
REPO_URL = "https://github.com/Artifex2002/wm_weight_space_experiments.git"
PROJECT_REF = "2ce606163917e5f9953a5842e61cb322c722c236"
EXPERIMENT_ID = "wall-pilot-001"
DRIVE_FOLDER = "/content/drive/MyDrive/wsp-stopper"
ATTEMPT_ID = "attempt-001"
ANALYSIS_ID = "diagnostics-001"  # New ID if you intentionally change settings.


## Setup
Select **Runtime → Change runtime type → GPU**. Authorize your own Drive.
If this is a fresh VM, setup downloads the pinned model assets again. Existing
Drive results remain in place. No API key is required.


In [ ]:
from pathlib import Path
import sys, subprocess, os, json, hashlib, shutil
from google.colab import drive
assert 'YOUR_' not in REPO_URL, 'Set REPO_URL to your repository URL'
assert '/' not in EXPERIMENT_ID and '..' not in EXPERIMENT_ID
# For a private repo, clone it manually using your preferred credentials first;
# never embed a GitHub token in REPO_URL or save one in notebook output.
drive.mount('/content/drive')
PROJECT = Path('/content/wsp-stopper')
if not (PROJECT/'.git').exists():
    subprocess.run(['git','clone',REPO_URL,str(PROJECT)],check=True)
else:
    assert subprocess.check_output(['git','-C',str(PROJECT),'remote','get-url','origin'],text=True).strip()==REPO_URL
subprocess.run(['git','-C',str(PROJECT),'checkout',PROJECT_REF],check=True)
os.chdir(PROJECT)
PROJECT_COMMIT=subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip()
required=['core.py','cli.py','agent.py','packet.py','inspection.py','wall_setup.py','cem_hook.py','upstream_patch.py']
assert all((PROJECT/'src/wsp_stopper'/f).exists() for f in required), 'Commit the repaired complete starter first'
# Retain Colab's preinstalled matched torch/torchvision pair; do not install the
# upstream all-simulator pyproject (which declares Python 3.10 only).
subprocess.run([sys.executable,'-m','pip','install','--quiet',
    'einops==0.8.1','tensordict==0.9.1','gym==0.23.1','omegaconf==2.3.0',
    'PyYAML>=6,<7','scipy','matplotlib','tqdm','pandas'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-e','.', '--no-deps','--quiet'],check=True)
sys.path.insert(0,str(PROJECT/'src'))
import torch, torchvision, tensordict
assert tuple(int(x) for x in torch.__version__.split('+')[0].split('.')[:2]) >= (2,7)
assert torch.cuda.is_available(), 'Runtime > Change runtime type > GPU, then rerun'
print('Python:',sys.version.split()[0], 'torch:',torch.__version__,'vision:',torchvision.__version__)
print('GPU:',torch.cuda.get_device_name(0),'project:',PROJECT_COMMIT)
EXPERIMENT = Path(DRIVE_FOLDER)/EXPERIMENT_ID
EXPERIMENT.mkdir(parents=True,exist_ok=True)
WORK = PROJECT/'assets/wall-colab-v1'
BASE_PACKET = PROJECT/'packets/wall-colab-base-v1'


In [ ]:
# This exact helper is embedded in both notebooks: no extra repo files required.
RUNTIME = Path('/content/wsp_wall_notebook_runtime.py')
RUNTIME_SOURCE = '"""Shared Colab runtime. Embedded byte-identically in both notebooks.\nRuns pinned upstream model/planner classes with a small dependency surface.\nNo evaluation files are included in the generator\'s allowlisted packet.\n"""\nimport argparse\nimport ast\nimport hashlib\nimport json\nimport math\nimport os\nfrom pathlib import Path\nimport random\nimport sys\nimport time\nimport types\nfrom typing import Callable, List, Union\nfrom abc import ABC, abstractmethod\nfrom types import SimpleNamespace\n\nimport numpy as np\nimport torch\nfrom torch import nn\nimport torch.nn.functional as F\nfrom einops import rearrange, repeat\nfrom tensordict import TensorDict\nimport yaml\nfrom wsp_stopper.core import digest, write_json\n\n\ndef definitions(path, names, namespace=None):\n    """Execute only named classes from pinned trusted source, no unrelated imports.\n    This keeps their method bodies verbatim and avoids simulator/training imports.\n    """\n    tree = ast.parse(Path(path).read_text())\n    selected = [n for n in tree.body if isinstance(n, ast.ClassDef) and n.name in names]\n    if {n.name for n in selected} != set(names):\n        raise ValueError(f\'Missing classes in {path}\')\n    env = dict(globals())\n    if namespace: env.update(namespace)\n    env[\'__name__\'] = \'wsp_notebook_upstream\'\n    exec(compile(ast.Module(body=selected,type_ignores=[]),str(path),\'exec\'),env)\n    return [env[name] for name in names]\n\n\ndef seed_all(seed):\n    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)\n    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)\n\n\ndef sync():\n    if torch.cuda.is_available(): torch.cuda.synchronize()\n\n\ndef configure(work):\n    work=Path(work).resolve()\n    up=work/\'jepa-wms\'\n    sys.path.insert(0,str(up))\n    os.environ[\'WSP_DINOV2_SOURCE\']=str(work/\'dinov2\')\n    os.environ[\'WSP_DINOV2_WEIGHTS\']=str(work/\'encoder.pth\')\n    torch.set_num_threads(2)\n    torch.backends.cuda.matmul.allow_tf32=False\n    torch.backends.cudnn.allow_tf32=False\n    torch.backends.cudnn.benchmark=False\n    return work,up\n\n\ndef make_environment(up):\n    # Direct DotWall construction avoids unrelated mujoco/robocasa/dataset imports.\n    from evals.simu_env_planning.envs.wall_env.envs.wall import DotWall\n    source=ast.parse((up/\'evals/simu_env_planning/envs/wall_gym_wrap.py\').read_text())\n    expr=next(n.value for n in source.body if isinstance(n,ast.Assign)\n        and any(isinstance(t,ast.Name) and t.id==\'DEFAULT_CFG\' for t in n.targets))\n    from evals.simu_env_planning.envs.wall_env.data.wall import WallDatasetConfig\n    cfg=eval(compile(ast.Expression(expr),\'wall_default_cfg\',\'eval\'),{\'WallDatasetConfig\':WallDatasetConfig})\n    cfg.device=\'cpu\'\n    # Match wrapper default layout generation; generate_random_state alternates travel direction.\n    return DotWall(rng=np.random.default_rng(0),wall_config=cfg,device=\'cpu\')\n\n\ndef create_tasks(work, output, count=10, seed_start=1000):\n    _,up=configure(work)\n    if Path(output).exists(): raise ValueError(\'Task manifest exists; reuse it instead of overwriting\')\n    seed_all(123)\n    env=make_environment(up)\n    episodes=[]\n    for i in range(count):\n        s,g=env.generate_random_state(seed_start+i)\n        episodes.append(dict(id=i,environment_seed=seed_start+i,planner_seed=900000+i,\n            start=s.tolist(),goal=g.tolist(),wall_x=float(env.wall_x),door_y=float(env.hole_y)))\n    write_json(output,dict(schema=1,task=\'Wall fixed layout cross-room navigation\',episodes=episodes))\n\n\nclass Preprocessor:\n    def __init__(self, stats, device):\n        from torchvision.transforms import Normalize\n        self.mean=torch.tensor(stats[\'proprio_mean\'])\n        self.std=torch.tensor(stats[\'proprio_std\'])\n        self.transform=Normalize([.5]*3,[.5]*3)\n    def normalize_proprios(self,x): return (x-self.mean)/self.std\n\n\ndef load_model(work,device):\n    _,up=configure(work)\n    # These modules have only small torch/einops dependencies.\n    from app.plan_common.models.vit import ViTPredictor\n    from app.plan_common.models.prop_embedding import ProprioceptiveEmbedding\n    from app.plan_common.models.dino import DinoEncoder\n    from app.vjepa_wm.video_wm import VideoWM\n    cfg=yaml.safe_load((work/\'wall.eval.yaml\').read_text())\n    kwargs=cfg[\'model_kwargs\'][\'pretrain_kwargs\']\n    stats_ast=ast.parse((up/\'app/plan_common/datasets/__init__.py\').read_text())\n    stats=next(ast.literal_eval(n.value)[\'wall\'] for n in stats_ast.body if isinstance(n,ast.Assign)\n        and any(isinstance(t,ast.Name) and t.id==\'DATA_STATS\' for t in n.targets))\n    encoder=DinoEncoder(\'dinov2_vits14\',\'x_norm_patchtokens\').to(device)\n    predictor=ViTPredictor(depth=6,heads=16,mlp_dim=2048,dropout=.1,\n        num_patches=256,num_frames=4,dim=404,use_sdpa=True).to(device)\n    def embedding(dim):\n        return ProprioceptiveEmbedding(num_frames=4,tubelet_size=1,tokens_per_step=1,\n            in_chans=dim,embed_dim=10,shift_input=False,mlp_dims=[10],use_mlp=False).to(device)\n    action,prop=embedding(10),embedding(2)\n    ckpt=torch.load(work/\'wall.pth.tar\',map_location=\'cpu\',weights_only=True)\n    report={}\n    for name,model in [(\'predictor\',predictor),(\'action_encoder\',action),(\'proprio_encoder\',prop)]:\n        state={k.replace(\'module.\',\'\'):v for k,v in ckpt[name].items()}\n        state={k.replace(\'state_encoder.weight\',\'proprio_encoder.weight\').replace(\'state_encoder.bias\',\'proprio_encoder.bias\'):v for k,v in state.items()}\n        # Some upstream versions omit regenerated causal masks from state_dict.\n        msg=model.load_state_dict(state,strict=False)\n        allowed=[k for k in msg.missing_keys if k.endswith(\'.bias\') and \'transformer.layers\' in k]\n        if msg.unexpected_keys or set(msg.missing_keys)-set(allowed):\n            raise ValueError(f\'Checkpoint mismatch: {name}: {msg}\')\n        report[name]=dict(missing_regenerated_masks=allowed,unexpected=msg.unexpected_keys)\n    video=VideoWM(encoder=encoder,predictor=predictor,action_encoder=action,proprio_encoder=prop,\n        enc_type=\'dino\',pred_type=\'dino_wm\',grid_size=16,tubelet_size_enc=1,img_size=224,\n        action_dim=10,proprio_dim=2,action_tokens=0,proprio_tokens=0,use_action=True,use_proprio=True,\n        action_skip=1,frameskip=5,action_conditioning=\'feature\',proprio_encoding=\'feature\',\n        action_encoder_inpred=False,proprio_encoder_inpred=False,batchify_video=True,\n        dup_image=False,normalize_reps=False,device=device,heads={},cfgs_loss={\'proprio_loss\':True})\n    Wrapper,=definitions(up/\'app/vjepa_wm/modelcustom/simu_env_planning/vit_enc_preds.py\',[\'EncPredWM\'])\n    wrapped=Wrapper(video,10,Preprocessor(stats,device),ctxt_window=2).to(device).eval()\n    report[\'encoder\']=\'strict=True, local pinned source and tensors\'\n    return wrapped,stats,report\n\n\ndef observation(env,state,device):\n    from torchvision.transforms import Resize\n    obs,_=env.reset(location=torch.tensor(state,dtype=torch.float32))\n    image=Resize((224,224))(obs[\'visual\'])\n    return {\'visual\':image.unsqueeze(0).unsqueeze(0).to(device),\n            \'proprio\':obs[\'proprio\'].view(1,1,2).to(device)}\n\n\ndef make_planner(work,model,z_goal,iterations,population,elites,chunk,planner_seed,device):\n    up=work/\'jepa-wms\'\n    objective_file=up/\'evals/simu_env_planning/planning/planning/objectives.py\'\n    _,Objective=definitions(objective_file,[\'BaseMPCObjective\',\'ReprTargetDistMPCObjective\'])\n    planner_file=up/\'evals/simu_env_planning/planning/planning/planner.py\'\n    _,_,CEM=definitions(planner_file,[\'PlanningResult\',\'Planner\',\'CEMPlanner\'],\n        {\'NamedTuple\':__import__(\'typing\').NamedTuple,\'objectives\':SimpleNamespace(BaseMPCObjective=object)})\n    counts={\'candidate_transition_evals\':0,\'mean_transition_evals\':0,\'candidate_sequences\':0}\n    def counted_unroll(z,act_suffix=None):\n        counts[\'mean_transition_evals\']+=act_suffix.shape[0]*act_suffix.shape[1]\n        return model.unroll(z,act_suffix=act_suffix)\n    generator=torch.Generator(device=device).manual_seed(planner_seed)\n    p=CEM(counted_unroll,iterations=iterations,num_samples=population,num_elites=elites,\n        horizon=6,action_dim=10,var_scale=1,num_act_stepped=6,\n        local_generator=generator,decode_each_iteration=False,distribute_planner=False)\n    p.device=torch.device(device)\n    p.objective=Objective(None,z_goal,sum_all_diffs=False,alpha=.1)\n    # Identical microbatching for baseline and WSP. No RNG is consumed by model eval.\n    def costs(self,actions,z):\n        vals=[]\n        for j in range(0,actions.shape[1],chunk):\n            a=actions[:,j:j+chunk]\n            predicted=model.unroll(z,act_suffix=a)\n            vals.append(self.objective(predicted,a))\n            counts[\'candidate_transition_evals\']+=a.shape[0]*a.shape[1]\n            counts[\'candidate_sequences\']+=a.shape[1]\n        return torch.cat(vals)\n    p.cost_function=types.MethodType(costs,p)\n    return p,counts\n\n\ndef evaluate(work,tasks_path,out,iterations=30,population=300,elites=10,chunk=16,weights=None,allow_cpu=False):\n    if Path(out).exists(): raise ValueError(\'Result exists; use a new filename\')\n    if weights and iterations!=30: raise ValueError(\'Controller requires cap 30\')\n    if not (2<=elites<=population and chunk>=1 and 1<=iterations<=30): raise ValueError(\'Invalid budget\')\n    if not torch.cuda.is_available() and not allow_cpu: raise RuntimeError(\'Select a Colab GPU runtime\')\n    device=\'cuda:0\' if torch.cuda.is_available() else \'cpu\'\n    work,up=configure(work)\n    seed_all(123)\n    model,stats,load_report=load_model(work,device)\n    episodes=json.loads(Path(tasks_path).read_text())[\'episodes\']\n    amean=torch.tensor(stats[\'action_mean\']);astd=torch.tensor(stats[\'action_std\'])\n    if weights: os.environ[\'WSP_STOPPER_WEIGHTS\']=str(Path(weights).resolve())\n    else: os.environ.pop(\'WSP_STOPPER_WEIGHTS\',None)\n    # Warm up encoding/one rollout, without using any scored evaluation outcome.\n    seed_all(42);env=make_environment(up)\n    warm=model.encode(observation(env,[15.,20.],device))\n    with torch.inference_mode(): model.unroll(warm,torch.zeros(1,1,10,device=device))\n    sync()\n    rows=[]\n    for ep in episodes:\n        seed_all(ep[\'environment_seed\']);env=make_environment(up)\n        if float(env.wall_x)!=ep[\'wall_x\'] or float(env.hole_y)!=ep[\'door_y\']: raise ValueError(\'Wall layout mismatch\')\n        goal_obs=observation(env,ep[\'goal\'],device)\n        start_obs=observation(env,ep[\'start\'],device)\n        sync();begin=time.perf_counter()\n        with torch.inference_mode():\n            z=model.encode(start_obs);zg=model.encode(goal_obs)\n            planner,counters=make_planner(work,model,zg,iterations,population,elites,chunk,ep[\'planner_seed\'],device)\n            result=planner.plan(z)\n        sync();elapsed=time.perf_counter()-begin\n        raw=result.actions.detach().cpu().reshape(-1,2)*astd+amean\n        seed_all(ep[\'environment_seed\'])\n        env.reset(location=torch.tensor(ep[\'start\'],dtype=torch.float32))\n        positions=[ep[\'start\']]\n        any_success=False\n        for a in raw:\n            _,_,_,info=env.step(a)\n            pos=info[\'state\'].detach().cpu().numpy()\n            positions.append(pos.tolist())\n            any_success |= bool(np.linalg.norm(pos-np.asarray(ep[\'goal\']))<4.5)\n        distance=float(np.linalg.norm(np.asarray(positions[-1])-np.asarray(ep[\'goal\'])))\n        depth=int(planner.wsp_last_depth)\n        rows.append(dict(id=ep[\'id\'],success=distance<4.5,any_step_success=any_success,\n            final_distance=distance,depth=depth,planning_seconds=elapsed,\n            utility=float(distance<4.5)-.05*depth/30,\n            actions=raw.tolist(),positions=positions,stopper_trace=planner.wsp_last_trace,\n            **counters))\n        print(f"episode {ep[\'id\']}: success={distance<4.5}, depth={depth}, seconds={elapsed:.2f}",flush=True)\n        del result,planner,z,zg\n    config=dict(horizon=6,chunk=6,frameskip=5,raw_episode_steps=30,population=population,\n        elites=elites,microbatch=chunk,cap=30,proprio_weight=.1,success=\'final_position_distance < 4.5\')\n    result=dict(schema=1,tasks_sha256=digest(tasks_path),runtime_sha256=digest(__file__),\n        release_lock=json.loads(Path(\'config/wall.release.json\').read_text()),config=config,\n        fixed_iterations=iterations,weights_sha256=digest(weights) if weights else None,\n        model_load_report=load_report,torch_version=torch.__version__,\n        device=torch.cuda.get_device_name(0) if torch.cuda.is_available() else \'CPU\',episodes=rows)\n    write_json(out,result)\n    return result\n\n\ndef main():\n    p=argparse.ArgumentParser();sub=p.add_subparsers(dest=\'cmd\',required=True)\n    t=sub.add_parser(\'tasks\');t.add_argument(\'--work\',required=True);t.add_argument(\'--out\',required=True)\n    t.add_argument(\'--count\',type=int,default=10);t.add_argument(\'--seed-start\',type=int,default=1000)\n    e=sub.add_parser(\'evaluate\');e.add_argument(\'--work\',required=True);e.add_argument(\'--tasks\',required=True);e.add_argument(\'--out\',required=True)\n    e.add_argument(\'--iterations\',type=int,default=30);e.add_argument(\'--population\',type=int,default=300)\n    e.add_argument(\'--elites\',type=int,default=10);e.add_argument(\'--microbatch\',type=int,default=16)\n    e.add_argument(\'--weights\');e.add_argument(\'--allow-cpu\',action=\'store_true\')\n    a=p.parse_args()\n    if a.cmd==\'tasks\': create_tasks(a.work,a.out,a.count,a.seed_start)\n    else: evaluate(a.work,a.tasks,a.out,a.iterations,a.population,a.elites,a.microbatch,a.weights,a.allow_cpu)\n\nif __name__==\'__main__\':main()\n'
RUNTIME.write_text(RUNTIME_SOURCE)
RUNTIME_SHA=hashlib.sha256(RUNTIME.read_bytes()).hexdigest()

def run_python(*args):
    subprocess.run([sys.executable,*map(str,args)],check=True,cwd=PROJECT)

from wsp_stopper.core import digest, write_json

def project_fingerprint():
    files=list((PROJECT/'src/wsp_stopper').glob('*.py'))+[PROJECT/'config/wall.release.json',PROJECT/'prompts/generate.md']
    return {str(p.relative_to(PROJECT)):digest(p) for p in sorted(files)}


In [ ]:
import copy, collections, zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from wsp_stopper.core import Stopper, validate_weights

assert all('/' not in s and '..' not in s for s in [ATTEMPT_ID, ANALYSIS_ID])
TASKS = EXPERIMENT/'tasks.json'
PROTOCOL = EXPERIMENT/'protocol.json'
ATTEMPT = EXPERIMENT/'generation'/ATTEMPT_ID
WEIGHTS = ATTEMPT/'weights.json'
CONTROL = EXPERIMENT/f'baseline_recheck_{ATTEMPT_ID}.json'
ADAPTIVE = EXPERIMENT/f'wsp_{ATTEMPT_ID}.json'
required = [TASKS, PROTOCOL, WEIGHTS, ATTEMPT/'run.json',
            ATTEMPT/'events.jsonl', CONTROL, ADAPTIVE]
missing = [str(p) for p in required if not p.is_file()]
assert not missing, f'Finish notebook 02 or copy the complete results into this account: {missing}'
protocol = json.loads(PROTOCOL.read_text())
assert protocol['project_commit'] == PROJECT_COMMIT
assert protocol['project_files'] == project_fingerprint()
assert protocol['runtime_sha256'] == RUNTIME_SHA
assert protocol['tasks_sha256'] == digest(TASKS)
WEIGHTS_SHA = digest(WEIGHTS)
validate_weights(json.loads(WEIGHTS.read_text()))
fixed = json.loads(CONTROL.read_text())
adaptive = json.loads(ADAPTIVE.read_text())
for data, sha in [(fixed, None), (adaptive, WEIGHTS_SHA)]:
    assert data['tasks_sha256'] == digest(TASKS)
    assert data['runtime_sha256'] == RUNTIME_SHA
    assert data['weights_sha256'] == sha
    assert data['release_lock'] == protocol['checkpoint_lock']
assert fixed['config'] == adaptive['config']
assert fixed['fixed_iterations'] == adaptive['fixed_iterations'] == 30
assert [e['id'] for e in fixed['episodes']] == [e['id'] for e in adaptive['episodes']]
POPULATION = protocol['population']
ELITES = protocol['elites']
MICROBATCH = protocol['microbatch']
ANALYSIS = EXPERIMENT/'analysis'/ANALYSIS_ID
ANALYSIS.mkdir(parents=True, exist_ok=True)

def frozen_json(path, value):
    path = Path(path)
    if path.exists():
        assert json.loads(path.read_text()) == value, f'Existing settings differ: {path}; choose a new ANALYSIS_ID'
    else:
        write_json(path, value)
    return path

frozen_json(ANALYSIS/'provenance.json', dict(
    project_commit=PROJECT_COMMIT, project_files=project_fingerprint(),
    runtime_sha256=RUNTIME_SHA, weights_sha256=WEIGHTS_SHA,
    original_tasks_sha256=digest(TASKS), checkpoint_lock=protocol['checkpoint_lock'],
    population=POPULATION, elites=ELITES, microbatch=MICROBATCH,
    api_generation='none; existing accepted controller reused'))
print('Original controller preserved:', WEIGHTS_SHA)
print('Diagnostics:', ANALYSIS)
print('Current GPU:', torch.cuda.get_device_name(0))


## 1. Audit generation and reproduce the recorded decisions — no GPU planning

The audit reports actual inspection calls, tensor reads, and API token usage.
The trace table reconstructs the **STOP minus CONTINUE logit margin** from the
accepted weights and recorded inputs. Positive means STOP, subject to the
mandatory guards (continue before iteration 3, stop at iteration 30).

A decision trace explains what the network computed. It cannot prove that a
particular iteration was optimal or that Astra diagnosed model error correctly.


In [ ]:
run_record = json.loads((ATTEMPT/'run.json').read_text())
events = [json.loads(line) for line in (ATTEMPT/'events.jsonl').read_text().splitlines() if line.strip()]
inspections = [e['value'] for e in events if e.get('kind') == 'tool']
responses = [e['value'] for e in events if e.get('kind') == 'response']
counts = collections.Counter(e['name'] for e in inspections)
usage = dict(input_tokens=0, output_tokens=0, reasoning_tokens=0)
for response in responses:
    u = response.get('usage') or {}
    usage['input_tokens'] += u.get('input_tokens', 0)
    usage['output_tokens'] += u.get('output_tokens', 0)
    usage['reasoning_tokens'] += (u.get('output_tokens_details') or {}).get('reasoning_tokens', 0)
audit = dict(model=run_record['model'], reasoning_effort=run_record.get('reasoning_effort'),
    inspection_calls=len(inspections), inspection_counts=dict(counts),
    sources_read=sorted({e['arguments']['path'] for e in inspections
                         if e['name']=='read_source' and isinstance(e.get('arguments'),dict)}),
    tensors_read=sorted({e['arguments']['name'] for e in inspections
                         if e['name']=='read_tensor' and isinstance(e.get('arguments'),dict)}),
    api_usage=usage, weights_sha256=WEIGHTS_SHA)
assert run_record['model']
print(json.dumps(audit, indent=2))
print('Reasoning tokens are included in output_tokens; do not add them again.')
call_table = pd.DataFrame([dict(call=i+1, tool=e['name'],
    arguments=json.dumps(e.get('arguments')), result_chars=len(e.get('output','')))
    for i,e in enumerate(inspections)])
display(call_table)
call_table.to_csv(ANALYSIS/'inspection_calls.csv', index=False)
write_json(ANALYSIS/'audit_summary.json', audit)
if counts['read_tensor'] == 0:
    print('No raw tensor values were requested. Inspect statistics/spectrum calls separately before claiming weight use.')

w = validate_weights(json.loads(WEIGHTS.read_text()))
stopper = Stopper(json.loads(WEIGHTS.read_text()))
feature_names = ['iteration_fraction','relative_improvement','smoothed_improvement',
    'relative_elite_cost','elite_best_gap','elite_cost_spread','proposal_sigma','first_action_change']
rows = []
for ep in adaptive['episodes']:
    trace = ep['stopper_trace']
    assert len(trace) == ep['depth']
    for entry in trace:
        x = np.asarray(entry['features'], dtype=np.float32)
        logits = w['W2'] @ np.maximum(0, w['W1'] @ x + w['b1']) + w['b2']
        decision = stopper.stop(x, entry['k'])
        assert decision == entry['stop'], 'Recorded decision differs from accepted controller'
        rows.append(dict(episode=ep['id'], k=entry['k'], stop=decision,
            forced_by_cap=entry['k']==30, margin=float(logits[1]-logits[0]),
            success=ep['success'], **dict(zip(feature_names,x.tolist()))))
trace_df = pd.DataFrame(rows)
trace_df.to_csv(ANALYSIS/'stopper_trace.csv', index=False)
display(trace_df.groupby('episode', sort=False).tail(1))
early_ids = trace_df.loc[trace_df.stop & ~trace_df.forced_by_cap, 'episode'].unique()
if len(early_ids):
    display(trace_df[trace_df.episode.isin(early_ids)].groupby('episode',sort=False).tail(6))
else:
    print('No episode stopped before the cap.')
fig, ax = plt.subplots(figsize=(9,4))
for eid, group in trace_df.groupby('episode'):
    ax.plot(group.k, group.margin, label=f'episode {eid}')
ax.axhline(0, color='black', linestyle='--'); ax.legend()
ax.set(xlabel='Completed CEM iteration', ylabel='STOP minus CONTINUE logit', title='Frozen controller decisions')
fig.tight_layout(); fig.savefig(ANALYSIS/'decision_margins.png', dpi=150); plt.show()


## Prepare GPU evaluation and checked resumption

This step downloads/verifies assets and freezes the base packet to generate the runtime configuration and apply pinned source patches, with no planning or generation. Child-process output is streamed and retained in process_logs/. All following
jobs use the original population, elites, microbatch, horizon, objective and
checkpoint. Each completed JSON is reused only after identity/configuration checks.
Each result file is written at the end of that job: interruption may require
rerunning the interrupted job, while earlier completed jobs remain reusable.


In [ ]:
# Surface child-process errors in Colab and retain their complete logs.
from datetime import datetime, timezone
from collections import deque
import uuid

def run_python(*args):
    log_dir = ANALYSIS/'process_logs'
    log_dir.mkdir(parents=True, exist_ok=True)
    stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
    log_path = log_dir/f'{stamp}-{uuid.uuid4().hex[:8]}.txt'
    command = [sys.executable, *map(str,args)]
    tail = deque(maxlen=100)
    with log_path.open('w') as log:
        log.write(json.dumps(command)+'\n'); log.flush()
        with subprocess.Popen(command,cwd=PROJECT,stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT,text=True,bufsize=1) as process:
            for line in process.stdout:
                print(line,end='',flush=True)
                log.write(line); log.flush(); tail.append(line)
            return_code = process.wait()
    if return_code:
        raise RuntimeError(f'Subprocess exited {return_code}. Full log: {log_path}\n'
                           + ''.join(tail)[-6000:])

# prepare downloads tensors; freeze also creates wall.eval.yaml and patches the planner.
run_python('-m','wsp_stopper.wall_setup','prepare','--work',WORK)
if not BASE_PACKET.exists():
    run_python('-m','wsp_stopper.wall_setup','freeze','--work',WORK,'--out',BASE_PACKET)
run_python('-m','wsp_stopper.cli','verify',BASE_PACKET)
assert (WORK/'wall.eval.yaml').is_file(), 'Missing runtime config: freeze did not complete'
assert (WORK/'patch-receipt.json').is_file(), 'Missing planner/encoder patch receipt'
from wsp_stopper.upstream_patch import patch
patch(WORK/'jepa-wms', protocol['checkpoint_lock'])  # Idempotent; refuses unrecognized edits.
assert json.loads((PROJECT/'config/wall.release.json').read_text()) == protocol['checkpoint_lock']

def validate_result(data, tasks, depth, weights):
    assert data['tasks_sha256'] == digest(tasks)
    assert data['runtime_sha256'] == RUNTIME_SHA
    assert data['weights_sha256'] == (WEIGHTS_SHA if weights else None)
    assert data['fixed_iterations'] == depth
    assert data['release_lock'] == protocol['checkpoint_lock']
    assert data['config'] == fixed['config']
    assert data['device'] == torch.cuda.get_device_name(0)
    assert data['torch_version'] == torch.__version__
    expected = json.loads(Path(tasks).read_text())['episodes']
    assert [e['id'] for e in data['episodes']] == [e['id'] for e in expected]
    return data

def evaluate_saved(tasks, output, depth=30, weights=None):
    tasks, output = Path(tasks), Path(output)
    assert digest(WEIGHTS) == WEIGHTS_SHA, 'Controller changed'
    assert digest(RUNTIME) == RUNTIME_SHA
    assert project_fingerprint() == protocol['project_files']
    assert weights is None or Path(weights).resolve() == WEIGHTS.resolve()
    output.parent.mkdir(parents=True, exist_ok=True)
    if not output.exists():
        args = [RUNTIME, 'evaluate', '--work', WORK, '--tasks', tasks, '--out', output,
                '--iterations', depth, '--population', POPULATION,
                '--elites', ELITES, '--microbatch', MICROBATCH]
        if weights is not None: args += ['--weights', weights]
        run_python(*args)
    else:
        print('Reusing completed job:', output.name)
    result = validate_result(json.loads(output.read_text()), tasks, depth, weights)
    assert digest(WEIGHTS) == WEIGHTS_SHA
    return result

def summary_row(method, result):
    es = result['episodes']
    return dict(method=method, episodes=len(es), successes=sum(e['success'] for e in es),
        success_rate=float(np.mean([e['success'] for e in es])),
        mean_depth=float(np.mean([e['depth'] for e in es])),
        mean_seconds=float(np.mean([e['planning_seconds'] for e in es])),
        mean_utility=float(np.mean([e['utility'] for e in es])))


## 2. Fixed-depth sweep on the original six episodes — exploratory

Set `RUN_DEPTH_SWEEP = True` and run the cell. The requested depths are declared
before any new sweep results. Existing fixed-30 and WSP scores are shown as the
pilot reference; **use the repeated-timing section for hardware comparisons**.

At your observed ~507 seconds per 30-iteration episode, new depths 3, 5, 10, 15,
20 and 25 total roughly **2.2 GPU hours** for six episodes on comparable hardware.
This estimate excludes setup/load overhead. Jobs save separately by depth.
The controller is never regenerated. Save the curve even if a fixed budget wins.


In [ ]:
RUN_DEPTH_SWEEP = False
DEPTHS = [3, 5, 10, 15, 20, 25, 30]
if RUN_DEPTH_SWEEP:
    frozen_json(ANALYSIS/'sweep_settings.json', dict(depths=DEPTHS,tasks_sha256=digest(TASKS)))
    results = []
    for depth in DEPTHS:
        result = fixed if depth==30 else evaluate_saved(TASKS, ANALYSIS/'sweep'/f'fixed_k{depth}.json', depth)
        results.append(summary_row(f'fixed_{depth}', result))
    results.append(summary_row('frozen_wsp', adaptive))
    sweep = pd.DataFrame(results)
    sweep.to_csv(ANALYSIS/'depth_sweep.csv', index=False)
    display(sweep)
    fig, axes = plt.subplots(1,2,figsize=(10,4))
    values = sweep[sweep.method!='frozen_wsp']
    axes[0].plot(DEPTHS, values.success_rate, 'o-')
    axes[1].plot(DEPTHS, values.mean_utility, 'o-')
    adaptive_ep_depth = sweep.iloc[-1].mean_depth
    axes[0].scatter([adaptive_ep_depth], [sweep.iloc[-1].success_rate], marker='*',s=150,label='WSP mean depth')
    axes[1].scatter([adaptive_ep_depth], [sweep.iloc[-1].mean_utility], marker='*',s=150,label='WSP mean depth')
    for ax, title in zip(axes,['Success rate','Mean utility']):
        ax.set(xlabel='CEM iterations (WSP: mean)',ylabel=title); ax.legend()
    axes[0].set_ylim(-.05,1.05)
    fig.tight_layout(); fig.savefig(ANALYSIS/'depth_sweep.png',dpi=150); plt.show()
else:
    print('Skipped. Set RUN_DEPTH_SWEEP=True when ready.')


## 3. New episodes with the same frozen controller

Set `RUN_HELDOUT = True`. This creates **12 new start/goal pairs** on the same
fixed wall layout, with environment seeds starting at 2000 and distinct planner
seeds starting at 1000000. Original coordinates and seeds are explicitly excluded.
The manifest is saved before evaluation and never overwritten.

Default comparison: fixed 30 versus the frozen controller. If the sweep motivates
one fixed budget, set `EXTRA_FIXED_DEPTH` **once, before running this section**;
it is recorded and evaluated on the same new episodes. Do not choose that budget
from these new-episode outcomes. This remains a post-hoc validation pilot.

On comparable hardware, the default pair costs roughly **3.4 GPU hours** if WSP
mostly runs to the cap. Additional fixed budgets cost extra. Increase coverage in
a new analysis ID after assessing runtime; a larger sample is not a guarantee of
statistical power.


In [ ]:
RUN_HELDOUT = False
HELDOUT_COUNT = 12
HELDOUT_SEED_START = 2000
HELDOUT_PLANNER_SEED_START = 1000000
EXTRA_FIXED_DEPTH = None  # Optional: one depth chosen using ONLY the original pilot sweep.
if RUN_HELDOUT:
    assert HELDOUT_COUNT>=1
    assert EXTRA_FIXED_DEPTH is None or EXTRA_FIXED_DEPTH in range(1,30)
    frozen_json(ANALYSIS/'heldout_settings.json', dict(count=HELDOUT_COUNT,
        seed_start=HELDOUT_SEED_START,planner_seed_start=HELDOUT_PLANNER_SEED_START,
        extra_fixed_depth=EXTRA_FIXED_DEPTH))
    held_tasks = ANALYSIS/'heldout_tasks.json'
    if not held_tasks.exists():
        import importlib.util
        module_spec = importlib.util.spec_from_file_location('wsp_diagnostic_runtime', RUNTIME)
        runner = importlib.util.module_from_spec(module_spec); module_spec.loader.exec_module(runner)
        staged = ANALYSIS/'heldout_tasks.pending.json'
        if staged.exists():
            raise RuntimeError('Pending manifest exists from an interrupted setup. Inspect it before removing only that pending file.')
        runner.create_tasks(WORK, staged, count=HELDOUT_COUNT, seed_start=HELDOUT_SEED_START)
        new_manifest = json.loads(staged.read_text())
        for i, ep in enumerate(new_manifest['episodes']):
            ep['planner_seed'] = HELDOUT_PLANNER_SEED_START+i
        write_json(staged,new_manifest)
        # Rename only after all checks below, so evaluation never sees partial setup.
        candidate = new_manifest
    else:
        candidate = json.loads(held_tasks.read_text())
    old_eps = json.loads(TASKS.read_text())['episodes']
    new_eps = candidate['episodes']
    assert len(new_eps)==HELDOUT_COUNT
    assert [e['environment_seed'] for e in new_eps]==list(range(HELDOUT_SEED_START,HELDOUT_SEED_START+HELDOUT_COUNT))
    assert [e['planner_seed'] for e in new_eps]==list(range(HELDOUT_PLANNER_SEED_START,HELDOUT_PLANNER_SEED_START+HELDOUT_COUNT))
    for field in ['environment_seed','planner_seed']:
        assert not ({e[field] for e in old_eps} & {e[field] for e in new_eps})
    def task_key(e): return tuple(e['start']),tuple(e['goal']),e['wall_x'],e['door_y']
    assert not ({task_key(e) for e in old_eps} & {task_key(e) for e in new_eps})
    assert len({task_key(e) for e in new_eps})==len(new_eps)
    if not held_tasks.exists(): staged.rename(held_tasks)
    frozen_json(ANALYSIS/'heldout_manifest_receipt.json',dict(tasks_sha256=digest(held_tasks)))
    new_fixed = evaluate_saved(held_tasks,ANALYSIS/'heldout'/'fixed_k30.json')
    new_wsp = evaluate_saved(held_tasks,ANALYSIS/'heldout'/'frozen_wsp.json',weights=WEIGHTS)
    results = [summary_row('fixed_30',new_fixed),summary_row('frozen_wsp',new_wsp)]
    if EXTRA_FIXED_DEPTH is not None:
        extra = evaluate_saved(held_tasks,ANALYSIS/'heldout'/f'fixed_k{EXTRA_FIXED_DEPTH}.json',EXTRA_FIXED_DEPTH)
        results.append(summary_row(f'fixed_{EXTRA_FIXED_DEPTH}',extra))
    summary = pd.DataFrame(results); display(summary)
    summary.to_csv(ANALYSIS/'heldout_summary.csv',index=False)
    a = pd.DataFrame(new_fixed['episodes']).set_index('id')
    b = pd.DataFrame(new_wsp['episodes']).set_index('id')
    paired = pd.DataFrame(dict(fixed_success=a.success,wsp_success=b.success,
        fixed_depth=a.depth,wsp_depth=b.depth,fixed_distance=a.final_distance,wsp_distance=b.final_distance))
    paired.to_csv(ANALYSIS/'heldout_paired.csv'); display(paired)
    print('WSP wins:',int(((~a.success)&b.success).sum()),'WSP losses:',int((a.success&(~b.success)).sum()))
    print('Iteration saving:',1-b.depth.sum()/a.depth.sum())
    print('Same labels on a small sample do not prove equal success rates.')
else:
    print('Skipped. Set RUN_HELDOUT=True when ready.')


## 4. Repeat timing on a diagnostic subset, alternating execution order

Set `RUN_TIMING = True`. IDs **0, 4, 5** intentionally include the early-stop case
and two successful pilot cases. This is a diagnostic subset, not an unbiased
population estimate. Two repeats alternate baseline→WSP and WSP→baseline. Each
result uses a fresh model process and the runtime's standard warm-up. Loading and
warm-up are outside the recorded planning time. There are no concurrent GPU jobs.

Default cost: roughly **1.7 GPU hours** on comparable hardware. Four or more
repeats give a better view of variability; choose the count before starting this
section. Completed repeat jobs are cached. Use a new analysis ID if hardware or
software changes. Timing estimates below are descriptive, with no significance claim.


In [ ]:
RUN_TIMING = False
TIMING_EPISODE_IDS = [0,4,5]
TIMING_REPEATS = 2  # Even count gives balanced method order.
if RUN_TIMING:
    assert TIMING_REPEATS>=2 and TIMING_REPEATS%2==0
    frozen_json(ANALYSIS/'timing_settings.json',dict(episode_ids=TIMING_EPISODE_IDS,repeats=TIMING_REPEATS,
        device=torch.cuda.get_device_name(0),torch_version=torch.__version__,
        project_commit=PROJECT_COMMIT,runtime_sha256=RUNTIME_SHA))
    original_manifest = json.loads(TASKS.read_text())
    by_id = {e['id']:e for e in original_manifest['episodes']}
    timing_manifest = dict(original_manifest,episodes=[by_id[i] for i in TIMING_EPISODE_IDS])
    timing_tasks = frozen_json(ANALYSIS/'timing_tasks.json',timing_manifest)
    timing_rows = []
    for repeat in range(TIMING_REPEATS):
        methods = [('fixed_30',None),('wsp',WEIGHTS)]
        if repeat%2: methods.reverse()
        for order,(method,weights) in enumerate(methods):
            result = evaluate_saved(timing_tasks,ANALYSIS/'timing'/f'repeat_{repeat+1:02d}_{method}.json',weights=weights)
            for ep in result['episodes']:
                timing_rows.append(dict(repeat=repeat+1,order=order+1,method=method,
                    episode=ep['id'],seconds=ep['planning_seconds'],depth=ep['depth'],success=ep['success']))
    timing = pd.DataFrame(timing_rows)
    timing.to_csv(ANALYSIS/'timing_measurements.csv',index=False)
    display(timing.groupby(['method','episode']).agg(mean_seconds=('seconds','mean'),
        std_seconds=('seconds','std'),mean_depth=('depth','mean')))
    totals = timing.groupby(['repeat','method']).seconds.sum().unstack()
    totals['time_saving_fraction']=1-totals.wsp/totals.fixed_30
    display(totals)
    totals.to_csv(ANALYSIS/'timing_totals.csv')
    print('Aggregate time saving on THIS diagnostic subset:',1-totals.wsp.sum()/totals.fixed_30.sum())
    print('Depth/success consistency across repeats:')
    display(timing.groupby(['method','episode'])[['depth','success']].nunique())
else:
    print('Skipped. Set RUN_TIMING=True when ready.')


## Export actual files, including audit logs and controller — no GPU planning

Run this even if the expensive sections are skipped. This creates a ZIP from the
real files in Drive, avoiding the previous empty-directory export. It includes the
original handoff, controller, generation audit, evaluations, and any completed
analysis. It does not include model checkpoints, API credentials or notebook state.
Audit logs contain model responses and inspected source/weight excerpts; review
before posting them publicly.

After download, attach the ZIP for analysis. Keep the controller hash in your report.


In [ ]:
from google.colab import files
from datetime import datetime, timezone
export_files = []
for name in ['tasks.json','protocol.json','baseline_k30.json',
             f'baseline_recheck_{ATTEMPT_ID}.json',f'wsp_{ATTEMPT_ID}.json',
             f'comparison_{ATTEMPT_ID}.json',f'summary_{ATTEMPT_ID}.csv',f'paired_{ATTEMPT_ID}.csv']:
    path = EXPERIMENT/name
    if path.is_file(): export_files.append(path)
for folder in [ATTEMPT, ANALYSIS]:
    export_files.extend(p for p in folder.rglob('*') if p.is_file() and p.suffix in ['.json','.jsonl','.txt','.csv','.png'])
export_files = sorted(set(export_files))
assert all(p in export_files for p in [WEIGHTS,ATTEMPT/'run.json',ATTEMPT/'events.jsonl',CONTROL,ADAPTIVE])
assert digest(WEIGHTS)==WEIGHTS_SHA
stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
archive_path = Path('/content')/f'{EXPERIMENT_ID}-{ANALYSIS_ID}-{stamp}.zip'
with zipfile.ZipFile(archive_path,'w',zipfile.ZIP_DEFLATED) as archive:
    for path in export_files:
        archive.write(path,f'{EXPERIMENT_ID}/{path.relative_to(EXPERIMENT)}')
with zipfile.ZipFile(archive_path) as archive:
    entries=archive.infolist()
    assert len(entries)==len(export_files)>0 and all(not e.is_dir() for e in entries)
    assert archive.testzip() is None
print(f'Exported {len(export_files)} actual files, {archive_path.stat().st_size:,} bytes')
files.download(str(archive_path))
